# UIT DSC 2026 – LegalQA Hybrid RAG (BM25 + Dense FAISS + RRF + Reranker + LLM)

Notebook này triển khai trọn gói giải pháp Hybrid RAG tiên tiến cho cuộc thi UIT DSC 2026:

1. **Tự động nhận diện môi trường**: Kaggle (GPU T4/P100), Google Colab, hoặc Local.
2. **Tự tìm dataset**: `train.json`, `public-official.json` và `selected-contexts` (thư mục lồng `selected-contexts/selected-contexts` hoặc file zip).
3. **Hybrid Retrieval đa tầng**:
   - **BM25 Top-50** qua SQLite FTS5.
   - **Dense FAISS Top-50** qua `AITeamVN/Vietnamese_Embedding_v2`.
   - **RRF Fusion (k=60)** hợp nhất thứ hạng $\rightarrow$ chọn **Top-50 candidate chunks**.
   - **Cross-Encoder Reranker** qua `AITeamVN/Vietnamese_Reranker` $\rightarrow$ chọn **Top-3 chunks** chính xác nhất.
4. **LLM Generation**: Dùng `AITeamVN/Vi-Qwen2-1.5B-RAG` cùng SYSTEM_PROMPT & RAG_TEMPLATE.
5. **Hiển thị tiến trình realtime & tự động lưu checkpoint định kỳ**.
6. **Kiểm tra định dạng submission** và đóng gói `.zip` sẵn sàng nộp lên Codabench.

In [ ]:
from __future__ import annotations

import json
import os
import sqlite3
import statistics
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from IPython.display import FileLink, Markdown, display

# ===== CẤU HÌNH CHÍNH =====
REPO_URL = "https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git"

# Tự động nhận diện môi trường
IS_KAGGLE = Path("/kaggle").is_dir()
IS_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()

if IS_KAGGLE:
    REPO_DIR = Path("/kaggle/working/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/kaggle/input")
    WORK_DIR = Path("/kaggle/working/legalqa-run")
elif IS_COLAB:
    REPO_DIR = Path("/content/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/content")
    WORK_DIR = Path("/content/legalqa-run")
else:
    # Môi trường Local máy tính
    REPO_DIR = Path("./UIT_DSC_2026_LegalQA_baseline_v0.1").resolve()
    if not REPO_DIR.exists():
        REPO_DIR = Path(".").resolve()
    INPUT_ROOT = Path(".").resolve()
    WORK_DIR = REPO_DIR / "artifacts"

# Ghi đè đường dẫn thủ công nếu cần (để None để tự động quét):
DATASET_DIR = None
TRAIN_PATH = None
PUBLIC_PATH = None
CONTEXTS_PATH = None

# Cấu hình thực thi index
FORCE_REBUILD_INDEX = False
BUILD_DENSE_INDEX = True      # Dense là thành phần bắt buộc của cấu hình RAG mặc định
RUN_TESTS = True
RUN_VALIDATION = False       # Bật True nếu muốn đo kiểm tra trên Train
VALIDATION_LIMIT = 100       # 100 nhanh hơn; full baseline dùng 300

# Chế độ sinh: "rag", "hybrid_rag", "hybrid", "extractive", "knn"
MODE = "rag"                 # Khuyên dùng "rag" hoặc "hybrid_rag"
KNN_THRESHOLD = 0.72

# Cấu hình Hybrid Retrieval & Reranking
BM25_TOP_K = 50
DENSE_TOP_K = 50
RRF_K = 60
RRF_TOP_K = 50
RERANK_TOP_K = 3             # Top 3 chunks đưa vào LLM
DENSE_QUERY_MAX_LENGTH = 256
DENSE_DOCUMENT_MAX_LENGTH = 2048
RERANKER_MAX_LENGTH = 2304
ALLOW_RETRIEVAL_FALLBACK = False
EMBEDDING_MODEL = "AITeamVN/Vietnamese_Embedding_v2"
RERANKER_MODEL = "AITeamVN/Vietnamese_Reranker"
GENERATOR_MODEL = "AITeamVN/Vi-Qwen2-1.5B-RAG"

# Cấu hình LLM Generator
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
TOP_P = 0.9
MAX_INPUT_TOKENS = 7168
GENERATION_SEED = 2026
DEVICE = "cuda" if (IS_KAGGLE or IS_COLAB) else "auto"
CHECKPOINT_INTERVAL = 10     # Lưu checkpoint mỗi N câu

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Môi trường: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")
print(f"Python: {sys.version.split()[0]}")
print(f"Repo dir: {REPO_DIR}")
print(f"Working directory: {WORK_DIR}")
print(f"Pipeline: BM25({BM25_TOP_K}) + Dense({DENSE_TOP_K}) -> RRF({RRF_K}) Top-{RRF_TOP_K} -> Reranker Top-{RERANK_TOP_K} -> {GENERATOR_MODEL}")


## 1. Clone hoặc cập nhật mã nguồn

In [ ]:
def run(command: list[str], cwd: Path | None = None) -> None:
    """Chạy lệnh terminal và stream log realtime ra output cell."""
    print("$", " ".join(map(str, command)))
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        encoding="utf-8",
        errors="replace",
    )
    if process.stdout is not None:
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
        process.stdout.close()
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


if IS_KAGGLE or IS_COLAB:
    if (REPO_DIR / ".git").is_dir():
        run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR)
    else:
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            import shutil
            shutil.rmtree(REPO_DIR, ignore_errors=True)
        run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

try:
    commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR,
        text=True,
    ).strip()
    print(f"Mã nguồn tại commit: {commit} ({REPO_DIR})")
except Exception:
    print(f"Đang dùng local folder: {REPO_DIR}")


## 2. Cài đặt thư viện Deep Learning, FAISS & Reranker

In [ ]:
if "rag" in MODE:
    req_path = REPO_DIR / "requirements-generator.txt"
    if req_path.exists():
        run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_path)])
        print("Đã cài đặt requirements-generator.txt")
    else:
        run([sys.executable, "-m", "pip", "install", "-q", "torch", "transformers>=4.40.0", "accelerate", "sentencepiece", "tqdm", "faiss-cpu", "numpy"])
        print("Đã cài đặt torch, transformers, faiss, reranker dependencies")


## 3. Tự động nhận diện dữ liệu đầu vào (Hỗ trợ thư mục lồng nhau)

In [ ]:
def resolve_file(
    label: str,
    override: str | Path | None,
    accepted_names: set[str],
) -> Path:
    if override is not None:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"{label}: không tồn tại: {path}")
        return path

    search_roots = [
        Path(DATASET_DIR) if DATASET_DIR is not None else None,
        INPUT_ROOT,
        REPO_DIR,
        REPO_DIR.parent,
        Path("./data"),
        Path("."),
    ]
    accepted = {name.casefold() for name in accepted_names}
    for root in search_roots:
        if root is None or not root.exists():
            continue
        matches = sorted(
            path for path in root.rglob("*")
            if path.is_file() and path.name.casefold() in accepted
        )
        if matches:
            return matches[0]

    raise FileNotFoundError(
        f"Không tìm thấy {label}. Tên chấp nhận: {sorted(accepted_names)}"
    )


def resolve_contexts(override: str | Path | None) -> Path:
    if override is not None:
        p = Path(override)
        if p.exists():
            return p

    search_roots = [
        Path(DATASET_DIR) if DATASET_DIR is not None else None,
        INPUT_ROOT,
        REPO_DIR,
        REPO_DIR.parent,
        Path("./data"),
        Path("."),
    ]
    for root in search_roots:
        if root is None or not root.exists():
            continue
        # 1. Tìm thư mục lồng selected-contexts/selected-contexts hoặc thư mục chứa json
        for p in root.rglob("*selected-contexts*"):
            if p.is_dir():
                # Nếu có thư mục con lồng nhau
                nested = p / "selected-contexts"
                if nested.is_dir() and any(nested.glob("context_*.json")):
                    return nested
                if any(p.glob("context_*.json")) or any(p.rglob("context_*.json")):
                    return p
        # 2. Tìm file zip
        for p in root.rglob("*selected-contexts*.zip"):
            if p.is_file():
                return p
    raise FileNotFoundError("Không tìm thấy selected-contexts (zip hoặc thư mục).")


TRAIN_PATH = resolve_file("Train", TRAIN_PATH, {"train.json"})
PUBLIC_PATH = resolve_file(
    "Public",
    PUBLIC_PATH,
    {"public-official(1).json", "public-official.json", "public_official.json", "public_test.json"},
)
CONTEXTS_PATH = resolve_contexts(CONTEXTS_PATH)

for label, path in [
    ("Train", TRAIN_PATH),
    ("Public", PUBLIC_PATH),
    ("Contexts", CONTEXTS_PATH),
]:
    size_mb = path.stat().st_size / 1024**2 if path.is_file() else 0
    print(f"{label:8s}: {path} ({size_mb:.2f} MiB)")

with TRAIN_PATH.open(encoding="utf-8") as handle:
    train_data = json.load(handle)
with PUBLIC_PATH.open(encoding="utf-8") as handle:
    public_data = json.load(handle)

assert isinstance(train_data, dict) and len(train_data) > 0
assert isinstance(public_data, dict) and len(public_data) > 0
assert all(isinstance(item.get("question"), str) for item in train_data.values())
assert all(isinstance(item.get("question"), str) for item in public_data.values())
print(f"Schema OK — Train: {len(train_data):,}, Public: {len(public_data):,}")


## 4. Kiểm tra môi trường và chạy Unit Tests

In [ ]:
connection = sqlite3.connect(":memory:")
try:
    connection.execute("CREATE VIRTUAL TABLE fts_check USING fts5(text)")
finally:
    connection.close()
print("SQLite FTS5: OK")

if RUN_TESTS:
    run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
        cwd=REPO_DIR,
    )
else:
    print("Bỏ qua unit test theo cấu hình.")


## 5. Dựng BM25 index & Dense Vector FAISS Index

In [ ]:
DB_PATH = WORK_DIR / "legalqa.sqlite"
DENSE_INDEX_PATH = WORK_DIR / "legalqa_dense"


def index_is_ready(path: Path) -> bool:
    if not path.is_file():
        return False
    try:
        connection = sqlite3.connect(f"file:{path.resolve()}?mode=ro", uri=True)
        metadata = dict(connection.execute("SELECT key, value FROM metadata"))
        connection.close()
        return int(metadata.get("chunks", 0)) > 0 and int(metadata.get("train_samples", 0)) > 0
    except (sqlite3.Error, ValueError):
        return False


# 1. Dựng BM25 FTS5 index
ready = index_is_ready(DB_PATH)
if FORCE_REBUILD_INDEX or not ready:
    if DB_PATH.exists() and not ready:
        for suffix in ("", "-wal", "-shm"):
            candidate = Path(str(DB_PATH) + suffix)
            if candidate.exists() or candidate.is_symlink():
                candidate.unlink()
        print("Đã dọn index không hoàn chỉnh để xây lại sạch.")
    command = [
        sys.executable, "-m", "legalqa_baseline", "build-index",
        "--contexts", str(CONTEXTS_PATH),
        "--train", str(TRAIN_PATH),
        "--db", str(DB_PATH),
    ]
    if DB_PATH.exists():
        command.append("--force")
    run(command, cwd=REPO_DIR)
else:
    print(f"Tái sử dụng BM25 index hoàn chỉnh: {DB_PATH}")

# 2. Dựng Dense FAISS index (nếu bật BUILD_DENSE_INDEX)
meta_dense = DENSE_INDEX_PATH.with_suffix(".meta.json")
dense_vectors = DENSE_INDEX_PATH.with_suffix(".faiss")
dense_numpy = DENSE_INDEX_PATH.with_suffix(".npy")
dense_ready = meta_dense.is_file() and (dense_vectors.is_file() or dense_numpy.is_file())
if BUILD_DENSE_INDEX and (not dense_ready or FORCE_REBUILD_INDEX):
    dense_cmd = [
        sys.executable, "-m", "legalqa_baseline", "build-dense-index",
        "--contexts", str(CONTEXTS_PATH),
        "--dense-index", str(DENSE_INDEX_PATH),
        "--embedding-model", EMBEDDING_MODEL,
        "--embedding-max-length", str(DENSE_DOCUMENT_MAX_LENGTH),
        "--device", DEVICE,
    ]
    if FORCE_REBUILD_INDEX or any(p.exists() for p in (meta_dense, dense_vectors, dense_numpy)):
        dense_cmd.append("--force")
    run(dense_cmd, cwd=REPO_DIR)
elif dense_ready:
    print(f"Đã tìm thấy Dense Vector Index có sẵn: {DENSE_INDEX_PATH}")
else:
    raise FileNotFoundError("Dense index chưa sẵn sàng; bật BUILD_DENSE_INDEX hoặc cung cấp đủ metadata + vector file.")


## 6. Validation tùy chọn

In [ ]:
VALIDATION_PATH = WORK_DIR / f"validation_{VALIDATION_LIMIT}.json"
if RUN_VALIDATION:
    val_cmd = [
        sys.executable, "-m", "legalqa_baseline", "validate",
        "--train", str(TRAIN_PATH),
        "--db", str(DB_PATH),
        "--output", str(VALIDATION_PATH),
        "--limit", str(VALIDATION_LIMIT),
        "--modes", MODE,
        "--knn-threshold", str(KNN_THRESHOLD),
        "--bm25-top-k", str(BM25_TOP_K),
        "--dense-top-k", str(DENSE_TOP_K),
        "--rrf-k", str(RRF_K),
        "--rrf-top-k", str(RRF_TOP_K),
        "--rerank-top-k", str(RERANK_TOP_K),
        "--dense-query-max-length", str(DENSE_QUERY_MAX_LENGTH),
        "--reranker-max-length", str(RERANKER_MAX_LENGTH),
        "--embedding-model", str(EMBEDDING_MODEL),
        "--reranker-model", str(RERANKER_MODEL),
        "--generator-model", str(GENERATOR_MODEL),
        "--device", str(DEVICE),
    ]
    if dense_ready:
        val_cmd.extend(["--dense-index", str(DENSE_INDEX_PATH)])
    if ALLOW_RETRIEVAL_FALLBACK:
        val_cmd.append("--allow-retrieval-fallback")
    run(val_cmd, cwd=REPO_DIR)
    display(json.loads(VALIDATION_PATH.read_text(encoding="utf-8")))
else:
    print("Validation đang tắt — tiếp tục sinh Public submission.")


## 7. Sinh câu trả lời cho Public Test (Hybrid RAG + Realtime Progress)

In [ ]:
SUBMISSION_PATH = WORK_DIR / f"submission_{MODE}.json"

predict_cmd = [
    sys.executable, "-m", "legalqa_baseline", "predict",
    "--input", str(PUBLIC_PATH),
    "--db", str(DB_PATH),
    "--output", str(SUBMISSION_PATH),
    "--mode", MODE,
    "--knn-threshold", str(KNN_THRESHOLD),
    "--bm25-top-k", str(BM25_TOP_K),
    "--dense-top-k", str(DENSE_TOP_K),
    "--rrf-k", str(RRF_K),
    "--rrf-top-k", str(RRF_TOP_K),
    "--rerank-top-k", str(RERANK_TOP_K),
    "--dense-query-max-length", str(DENSE_QUERY_MAX_LENGTH),
    "--reranker-max-length", str(RERANKER_MAX_LENGTH),
    "--embedding-model", str(EMBEDDING_MODEL),
    "--reranker-model", str(RERANKER_MODEL),
    "--generator-model", str(GENERATOR_MODEL),
    "--device", str(DEVICE),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--temperature", str(TEMPERATURE),
    "--top-p", str(TOP_P),
    "--max-input-tokens", str(MAX_INPUT_TOKENS),
    "--generation-seed", str(GENERATION_SEED),
    "--resume",
    "--checkpoint-interval", str(CHECKPOINT_INTERVAL),
]

if dense_ready:
    predict_cmd.extend(["--dense-index", str(DENSE_INDEX_PATH)])
if ALLOW_RETRIEVAL_FALLBACK:
    predict_cmd.append("--allow-retrieval-fallback")

run(predict_cmd, cwd=REPO_DIR)
assert SUBMISSION_PATH.is_file(), f"Không tìm thấy file submission: {SUBMISSION_PATH}"
print(f"[✓] Đã tạo thành công submission: {SUBMISSION_PATH}")


## 8. Kiểm tra chất lượng và định dạng trước khi nộp

In [ ]:
prediction_data = json.loads(SUBMISSION_PATH.read_text(encoding="utf-8"))

assert set(prediction_data) == set(public_data), (
    f"ID mismatch: public={len(public_data)}, prediction={len(prediction_data)}"
)
assert all(
    isinstance(item, dict) and set(item) == {"answer"}
    for item in prediction_data.values()
), "Mỗi prediction phải có đúng một trường answer"
answers = [item["answer"] for item in prediction_data.values()]
assert all(isinstance(answer, str) and answer.strip() for answer in answers), (
    "Phát hiện answer rỗng/null"
)
lengths = sorted(len(answer.split()) for answer in answers)
quality_report = {
    "ids": len(prediction_data),
    "empty_answers": sum(not answer.strip() for answer in answers),
    "words_min": min(lengths),
    "words_median": statistics.median(lengths),
    "words_p90": lengths[round(0.9 * (len(lengths) - 1))],
    "words_max": max(lengths),
    "file_mib": round(SUBMISSION_PATH.stat().st_size / 1024**2, 2),
}
print(json.dumps(quality_report, ensure_ascii=False, indent=2))
print("Submission schema: OK")


## 9. Xem thử kết quả và tải submission

In [ ]:
for sample_id in list(public_data)[:3]:
    question = public_data[sample_id]["question"]
    answer = prediction_data[sample_id]["answer"]
    preview = answer[:1500] + ("…" if len(answer) > 1500 else "")
    display(Markdown(f"### ID {sample_id}\n**Câu hỏi:** {question}\n\n**Trả lời:** {preview}"))

SUBMISSION_ZIP = WORK_DIR / f"submission_{MODE}.zip"
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(SUBMISSION_PATH, arcname=SUBMISSION_PATH.name)

print("Tệp JSON dùng để kiểm tra trực tiếp; tệp ZIP dùng nếu Codabench yêu cầu upload archive.")
display(FileLink(str(SUBMISSION_PATH)))
display(FileLink(str(SUBMISSION_ZIP)))


## Hoàn tất

Tệp cuối nằm trong thư mục output của session. Bạn có thể tải trực tiếp file `.json` hoặc `.zip` để nộp lên hệ thống chấm thi Codabench.